# Caso 3: Robot FX avanzado para USD/MXN

Objetivo: consumir datos de mercado, construir un modelo direccional, estimar probabilidad de alza del USD/MXN y generar un dashboard ejecutivo con agente IA dinámico.

## Qué produce este notebook

- Descarga de datos con `yfinance` o respaldo sintético.
- Modelo de clasificación para estimar si el USD/MXN sube en 7 días.
- Métricas de clasificación e importancia de variables.
- Dashboard HTML estilo magIA/Bloomberg.
- Agente IA renderizado como chat dinámico con señal, interpretación y preguntas rápidas.

In [ ]:
# Instalación mínima para Colab
# Si ya están instaladas, esta celda no causa problema.
!pip -q install pandas numpy scikit-learn plotly openpyxl yfinance

import os
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from IPython.display import HTML, display, IFrame

# Opcional: montar Google Drive en Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/Corporate_AI_Lab'
    os.makedirs(BASE_DIR, exist_ok=True)
except Exception:
    BASE_DIR = '.'

print('Carpeta de trabajo:', BASE_DIR)

def calcular_rmse(y_real, y_pred):
    """Calcula RMSE sin depender de mean_squared_error(..., squared=False)."""
    return np.sqrt(mean_squared_error(y_real, y_pred))


In [ ]:
# Cargar datos FX: intenta yfinance; si falla, usa base sintética
import warnings
warnings.filterwarnings('ignore')

source = 'CSV sintético'
try:
    import yfinance as yf
    raw = yf.download(['MXN=X','DX-Y.NYB','CL=F'], start='2021-01-01', progress=False, auto_adjust=False)
    if isinstance(raw.columns, pd.MultiIndex):
        close = raw['Close'].copy()
    else:
        close = raw.copy()
    close = close.rename(columns={'MXN=X':'USDMXN','DX-Y.NYB':'DXY_sintetico','CL=F':'WTI_sintetico'})
    data = close[['USDMXN','DXY_sintetico','WTI_sintetico']].dropna().reset_index()
    data = data.rename(columns={'Date':'fecha'})
    if len(data) < 250:
        raise ValueError('Datos insuficientes desde API')
    source = 'yfinance'
except Exception as e:
    print('No se pudo usar yfinance o los datos fueron insuficientes. Se usará respaldo sintético.')
    possible_paths = [
        os.path.join(BASE_DIR, 'tipo_cambio_sintetico.csv'),
        '/content/tipo_cambio_sintetico.csv',
        'tipo_cambio_sintetico.csv'
    ]
    path = next((p for p in possible_paths if os.path.exists(p)), None)
    if path is None:
        raise FileNotFoundError('Sube tipo_cambio_sintetico.csv o guárdalo en Corporate_AI_Lab dentro de Drive')
    data = pd.read_csv(path)

data['fecha'] = pd.to_datetime(data['fecha'])
data = data.sort_values('fecha').reset_index(drop=True)
print('Fuente:', source)
print(data.shape)
data.head()


In [ ]:
# Ingeniería de variables financieras
# Se construyen rendimientos, volatilidad, medias móviles y variables macro sintéticas/mercado.

df = data.copy()
for col in ['USDMXN','DXY_sintetico','WTI_sintetico']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.dropna(subset=['USDMXN','DXY_sintetico','WTI_sintetico']).copy()
df['ret_1d'] = df['USDMXN'].pct_change()
df['ret_5d'] = df['USDMXN'].pct_change(5)
df['ret_10d'] = df['USDMXN'].pct_change(10)
df['vol_10d'] = df['ret_1d'].rolling(10).std()
df['vol_20d'] = df['ret_1d'].rolling(20).std()
df['ma_10'] = df['USDMXN'].rolling(10).mean()
df['ma_30'] = df['USDMXN'].rolling(30).mean()
df['ma_90'] = df['USDMXN'].rolling(90).mean()
df['dist_ma10'] = df['USDMXN'] / df['ma_10'] - 1
df['dist_ma30'] = df['USDMXN'] / df['ma_30'] - 1
df['dist_ma90'] = df['USDMXN'] / df['ma_90'] - 1
df['DXY_ret_1d'] = df['DXY_sintetico'].pct_change()
df['DXY_ret_5d'] = df['DXY_sintetico'].pct_change(5)
df['WTI_ret_1d'] = df['WTI_sintetico'].pct_change()
df['WTI_ret_5d'] = df['WTI_sintetico'].pct_change(5)
df['usdmxn_7d_futuro'] = df['USDMXN'].shift(-7)
df['sube_7d'] = (df['usdmxn_7d_futuro'] > df['USDMXN']).astype(int)

df_model = df.dropna().copy()
features = ['ret_1d','ret_5d','ret_10d','vol_10d','vol_20d','dist_ma10','dist_ma30','dist_ma90','DXY_ret_1d','DXY_ret_5d','WTI_ret_1d','WTI_ret_5d']
X = df_model[features]
y = df_model['sube_7d']

split_idx = int(len(df_model) * 0.80)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model = RandomForestClassifier(n_estimators=400, random_state=42, min_samples_leaf=4, class_weight='balanced', n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred, zero_division=0)
recall = recall_score(y_test, pred, zero_division=0)
f1 = f1_score(y_test, pred, zero_division=0)
cm = confusion_matrix(y_test, pred)

print(f'Accuracy: {accuracy:.3f}')
print(f'Precision: {precision:.3f}')
print(f'Recall: {recall:.3f}')
print(f'F1: {f1:.3f}')

backtest = df_model.iloc[split_idx:].copy()
backtest['prediccion'] = pred
backtest['prob_subida'] = proba
backtest[['fecha','USDMXN','sube_7d','prediccion','prob_subida']].tail()


In [ ]:
# Señal actual del robot FX
latest = df_model.iloc[-1].copy()
X_latest = pd.DataFrame([latest[features]])
prob_up = float(model.predict_proba(X_latest)[0,1])
last_fx = float(latest['USDMXN'])
vol_7d = float(latest['vol_20d'] * np.sqrt(7))
lower = last_fx * (1 - vol_7d)
upper = last_fx * (1 + vol_7d)

if prob_up >= 0.60:
    signal = 'ALCISTA'
    action = 'mantener cobertura compradora o esperar confirmación técnica antes de aumentar exposición al dólar.'
elif prob_up <= 0.40:
    signal = 'BAJISTA'
    action = 'evitar compras anticipadas de USD y evaluar coberturas parciales si existen obligaciones en dólares.'
else:
    signal = 'NEUTRAL'
    action = 'mantener seguimiento; no hay ventaja estadística suficiente para tomar una posición direccional fuerte.'

important = pd.DataFrame({'variable': features, 'importancia': model.feature_importances_}).sort_values('importancia', ascending=False)
top_driver = important.iloc[0]['variable']

agent_messages = [
    {
        'role': 'assistant',
        'title': 'Señal actual del robot',
        'text': f'La probabilidad estimada de alza del USD/MXN a 7 días es {prob_up:.1%}. La señal ejecutiva del modelo es {signal}.'
    },
    {
        'role': 'assistant',
        'title': 'Rango estadístico esperado',
        'text': f'Con base en la volatilidad reciente, el rango aproximado a 7 días se ubica entre {lower:.4f} y {upper:.4f} pesos por dólar.'
    },
    {
        'role': 'assistant',
        'title': 'Driver dominante',
        'text': f'La variable con mayor peso en el modelo es {top_driver}. Esto no implica causalidad, pero sí relevancia predictiva dentro del patrón histórico analizado.'
    },
    {
        'role': 'assistant',
        'title': 'Acción sugerida',
        'text': f'Se recomienda {action} Métricas de validación: accuracy {accuracy:.3f}, precision {precision:.3f}, recall {recall:.3f}, F1 {f1:.3f}.'
    }
]

quick_answers = {
    '¿Comprar dólares ahora?': f'La señal es {signal}. Si la decisión es corporativa, conviene combinar la probabilidad {prob_up:.1%} con vencimientos reales, exposición en USD y política de cobertura.',
    '¿Cuál es el riesgo clave?': 'El principal riesgo es interpretar el modelo como predicción exacta. Es una señal probabilística basada en patrones históricos, no una garantía de precio futuro.',
    '¿Qué debo monitorear?': 'Monitorea DXY, petróleo, volatilidad reciente, distancia contra medias móviles y eventos macro como tasas, inflación y decisiones de bancos centrales.'
}

print('Último USD/MXN:', last_fx)
print('Probabilidad de alza:', f'{prob_up:.1%}')
print('Señal:', signal)


In [ ]:
# Dashboard HTML avanzado: Robot FX + agente IA dinámico
import os, json

colors = {'bg':'#07111f','card':'#0d1d36','cyan':'#22d8ff','green':'#42f5b3','amber':'#ffbf69','red':'#ff4d6d','purple':'#9b5cff','text':'#eaf6ff'}

def layout_fig(fig, title=None, height=430):
    fig.update_layout(
        template='plotly_dark', height=height, paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=40,r=20,t=60,b=40), font=dict(color=colors['text'], family='Inter, Arial'),
        title=dict(text=title or '', x=0.02, xanchor='left', font=dict(size=18)),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.update_xaxes(gridcolor='rgba(255,255,255,.08)')
    fig.update_yaxes(gridcolor='rgba(255,255,255,.08)')
    return fig

plot_df = df_model.copy()
fig_price = go.Figure()
fig_price.add_trace(go.Scatter(x=plot_df['fecha'], y=plot_df['USDMXN'], mode='lines', name='USD/MXN', line=dict(color=colors['cyan'], width=2)))
fig_price.add_trace(go.Scatter(x=plot_df['fecha'], y=plot_df['ma_30'], mode='lines', name='MA 30', line=dict(color=colors['amber'], width=1.4)))
fig_price.add_trace(go.Scatter(x=plot_df['fecha'], y=plot_df['ma_90'], mode='lines', name='MA 90', line=dict(color=colors['purple'], width=1.4)))
layout_fig(fig_price, 'USD/MXN histórico con medias móviles')

fig_prob = go.Figure()
fig_prob.add_trace(go.Scatter(x=backtest['fecha'], y=backtest['prob_subida'], mode='lines', name='Probabilidad de alza', line=dict(color=colors['green'], width=2)))
fig_prob.add_hline(y=0.60, line_dash='dash', line_color=colors['green'], annotation_text='Sesgo alcista')
fig_prob.add_hline(y=0.40, line_dash='dash', line_color=colors['red'], annotation_text='Sesgo bajista')
layout_fig(fig_prob, 'Backtest de probabilidad direccional')

fig_macro = go.Figure()
fig_macro.add_trace(go.Scatter(x=plot_df['fecha'], y=plot_df['DXY_sintetico'], name='DXY', line=dict(color=colors['purple'])))
fig_macro.add_trace(go.Scatter(x=plot_df['fecha'], y=plot_df['WTI_sintetico'], name='WTI', yaxis='y2', line=dict(color=colors['amber'])))
fig_macro.update_layout(yaxis2=dict(overlaying='y', side='right', gridcolor='rgba(255,255,255,0)'))
layout_fig(fig_macro, 'Variables externas: DXY y WTI')

cm_df = pd.DataFrame(cm, index=['Real baja/no sube','Real sube'], columns=['Pred baja/no sube','Pred sube'])
fig_cm = px.imshow(cm_df, text_auto=True, color_continuous_scale=['#0d1d36','#22d8ff'])
layout_fig(fig_cm, 'Matriz de confusión')

fig_fi = px.bar(important.sort_values('importancia'), x='importancia', y='variable', orientation='h')
layout_fig(fig_fi, 'Variables más importantes del modelo')

last_180 = plot_df.tail(180).copy()
fig_ret = go.Figure()
fig_ret.add_trace(go.Bar(x=last_180['fecha'], y=last_180['ret_1d'], name='Retorno diario', marker_color=colors['cyan']))
fig_ret.add_trace(go.Scatter(x=last_180['fecha'], y=last_180['vol_20d'], name='Volatilidad 20d', yaxis='y2', line=dict(color=colors['red'])))
fig_ret.update_layout(yaxis2=dict(overlaying='y', side='right', tickformat='.2%'))
layout_fig(fig_ret, 'Retorno diario y volatilidad')

figs = {
    'price': fig_price.to_html(full_html=False, include_plotlyjs='cdn'),
    'prob': fig_prob.to_html(full_html=False, include_plotlyjs=False),
    'macro': fig_macro.to_html(full_html=False, include_plotlyjs=False),
    'cm': fig_cm.to_html(full_html=False, include_plotlyjs=False),
    'fi': fig_fi.to_html(full_html=False, include_plotlyjs=False),
    'ret': fig_ret.to_html(full_html=False, include_plotlyjs=False),
}

bt = backtest[['fecha','USDMXN','sube_7d','prediccion','prob_subida']].tail(60).copy()
bt['fecha'] = bt['fecha'].dt.strftime('%Y-%m-%d')
bt['prob_subida'] = bt['prob_subida'].map(lambda x: f'{x:.1%}')
bt['USDMXN'] = bt['USDMXN'].map(lambda x: f'{x:.4f}')
backtest_html = bt.to_html(index=False, classes='table', border=0)

risk_color = {'ALCISTA':'var(--green)','BAJISTA':'var(--red)','NEUTRAL':'var(--amber)'}.get(signal,'var(--cyan)')
kpi_html = f"""
<div class='kpi'><span>Último USD/MXN</span><strong>{last_fx:.4f}</strong><small>fuente: {source}</small></div>
<div class='kpi'><span>Prob. alza 7 días</span><strong>{prob_up:.1%}</strong><small>clasificación ML</small></div>
<div class='kpi'><span>Señal</span><strong style='color:{risk_color}'>{signal}</strong><small>umbral 40% / 60%</small></div>
<div class='kpi'><span>Accuracy</span><strong>{accuracy:.3f}</strong><small>validación temporal</small></div>
"""

agent_json = json.dumps(agent_messages, ensure_ascii=False)
quick_json = json.dumps(quick_answers, ensure_ascii=False)

html = f"""
<!DOCTYPE html>
<html lang='es'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>magIA | Robot FX USD/MXN</title>
<link rel='preconnect' href='https://fonts.googleapis.com'>
<link rel='preconnect' href='https://fonts.gstatic.com' crossorigin>
<link href='https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap' rel='stylesheet'>
<style>
:root {{ --bg:#050b14; --panel:#0b1628; --panel2:#0f2039; --line:rgba(34,216,255,.18); --text:#eaf6ff; --muted:#9db6cf; --cyan:#22d8ff; --green:#42f5b3; --amber:#ffbf69; --red:#ff4d6d; --purple:#9b5cff; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; font-family:Inter,Arial,sans-serif; background:radial-gradient(circle at top left, rgba(34,216,255,.14), transparent 32%), radial-gradient(circle at top right, rgba(155,92,255,.12), transparent 30%), linear-gradient(180deg,#050b14,#07111f 45%,#08182b); color:var(--text); }}
.header {{ padding:28px 34px; border-bottom:1px solid var(--line); position:sticky; top:0; z-index:10; backdrop-filter:blur(18px); background:rgba(5,11,20,.76); }}
.brand {{ display:flex; justify-content:space-between; align-items:center; gap:18px; }} .logo {{ font-size:26px; font-weight:800; letter-spacing:-.04em; }} .logo span {{ color:var(--cyan); }}
.badge {{ padding:10px 14px; border:1px solid var(--line); border-radius:999px; color:var(--cyan); background:rgba(34,216,255,.08); font-weight:700; }}
.hero {{ padding:34px; display:grid; grid-template-columns:1.1fr .9fr; gap:20px; }}
.card {{ background:linear-gradient(180deg,rgba(15,32,57,.94),rgba(9,18,33,.94)); border:1px solid var(--line); border-radius:24px; box-shadow:0 18px 50px rgba(0,0,0,.35); overflow:hidden; }}
.hero-card {{ padding:30px; position:relative; }} h1 {{ font-size:clamp(2.2rem,4vw,4.7rem); line-height:.98; margin:14px 0; letter-spacing:-.06em; }} p {{ color:var(--muted); line-height:1.65; }}
.kpis {{ display:grid; grid-template-columns:repeat(4,1fr); gap:14px; padding:0 34px 22px; }} .kpi {{ padding:18px; border:1px solid rgba(255,255,255,.07); border-radius:20px; background:rgba(255,255,255,.035); }} .kpi span {{ color:var(--muted); display:block; font-size:.88rem; }} .kpi strong {{ display:block; font-size:1.7rem; margin:7px 0; }} .kpi small {{ color:var(--cyan); }}
.tabs {{ display:flex; gap:10px; flex-wrap:wrap; padding:10px 34px 24px; }} .tab {{ cursor:pointer; border:1px solid var(--line); background:rgba(255,255,255,.03); color:var(--muted); padding:12px 16px; border-radius:999px; font-weight:700; }} .tab.active {{ background:rgba(34,216,255,.14); color:#fff; border-color:rgba(34,216,255,.45); }}
.section {{ display:none; padding:0 34px 34px; }} .section.active {{ display:block; }} .grid2 {{ display:grid; grid-template-columns:1fr 1fr; gap:18px; }} .grid3 {{ display:grid; grid-template-columns:1.1fr .9fr; gap:18px; }} .plot {{ padding:14px; }}
.table-wrap {{ overflow:auto; max-height:520px; }} table.table {{ width:100%; border-collapse:collapse; min-width:700px; }} .table th,.table td {{ padding:12px 14px; border-bottom:1px solid rgba(255,255,255,.07); text-align:left; font-size:.92rem; }} .table th {{ color:#8cecff; background:rgba(34,216,255,.06); position:sticky; top:0; }}
.agent {{ display:flex; flex-direction:column; height:620px; }} .agent-head {{ padding:18px; border-bottom:1px solid var(--line); display:flex; align-items:center; gap:12px; }} .orb {{ width:42px; height:42px; border-radius:50%; background:radial-gradient(circle,#fff, var(--cyan) 35%, var(--purple)); box-shadow:0 0 28px rgba(34,216,255,.55); animation:pulse 2.2s infinite; }} @keyframes pulse {{ 0%,100%{{transform:scale(1)}} 50%{{transform:scale(1.08)}} }}
.agent-body {{ padding:18px; flex:1; overflow:auto; display:flex; flex-direction:column; gap:14px; }} .msg {{ max-width:86%; padding:14px 16px; border-radius:18px; border:1px solid rgba(255,255,255,.08); background:rgba(255,255,255,.05); opacity:0; transform:translateY(8px); animation:appear .5s forwards; }} .msg strong {{ color:#fff; display:block; margin-bottom:6px; }} .msg p {{ margin:0; color:#cfe4ff; }} @keyframes appear {{ to{{opacity:1; transform:translateY(0)}} }}
.typing {{ display:flex; gap:6px; padding:12px 16px; width:max-content; border-radius:18px; background:rgba(34,216,255,.10); }} .typing i {{ width:7px; height:7px; border-radius:50%; background:var(--cyan); animation:bounce 1s infinite; }} .typing i:nth-child(2){{animation-delay:.15s}} .typing i:nth-child(3){{animation-delay:.3s}} @keyframes bounce {{ 0%,80%,100%{{opacity:.3}} 40%{{opacity:1}} }}
.quick {{ display:flex; gap:8px; flex-wrap:wrap; padding:0 18px 18px; }} .quick button {{ border:1px solid var(--line); border-radius:999px; background:rgba(34,216,255,.08); color:#eaf6ff; padding:10px 12px; cursor:pointer; font-weight:700; }} .footer {{ padding:24px 34px; color:var(--muted); border-top:1px solid var(--line); }}
@media(max-width:1000px) {{ .hero,.grid2,.grid3,.kpis{{grid-template-columns:1fr}} h1{{font-size:2.6rem}} }}
</style>
</head>
<body>
<header class='header'><div class='brand'><div class='logo'>mag<span>IA</span> · Robot FX USD/MXN</div><div class='badge'>Python · API · ML · Treasury Analytics</div></div></header>
<section class='hero'><div class='card hero-card'><div class='badge'>API + modelo direccional + agente IA</div><h1>Robot inteligente de tipo de cambio</h1><p>Estimación probabilística de sesgo del USD/MXN a 7 días usando variables técnicas y factores externos como DXY y WTI.</p></div><div class='card hero-card'><h2>Señal ejecutiva</h2><p><strong>Señal:</strong> {signal}</p><p><strong>Probabilidad de alza:</strong> {prob_up:.1%}</p><p><strong>Rango estimado:</strong> {lower:.4f} – {upper:.4f}</p><p><strong>Acción:</strong> {action}</p></div></section>
<section class='kpis'>{kpi_html}</section>
<nav class='tabs'><button class='tab active' data-tab='overview'>Resumen</button><button class='tab' data-tab='market'>Mercado</button><button class='tab' data-tab='model'>Modelo</button><button class='tab' data-tab='backtest'>Backtest</button><button class='tab' data-tab='agent'>Agente IA</button></nav>
<section id='overview' class='section active'><div class='grid2'><div class='card plot'>{figs['price']}</div><div class='card plot'>{figs['prob']}</div></div></section>
<section id='market' class='section'><div class='grid2'><div class='card plot'>{figs['macro']}</div><div class='card plot'>{figs['ret']}</div></div></section>
<section id='model' class='section'><div class='grid2'><div class='card plot'>{figs['fi']}</div><div class='card plot'>{figs['cm']}</div></div></section>
<section id='backtest' class='section'><div class='grid3'><div class='card plot'>{figs['prob']}</div><div class='card table-wrap'>{backtest_html}</div></div></section>
<section id='agent' class='section'><div class='grid3'><div class='card agent'><div class='agent-head'><div class='orb'></div><div><strong>Agente IA de Tesorería FX</strong><p style='margin:4px 0 0'>Interpreta señal, riesgo y variables críticas.</p></div></div><div class='agent-body' id='agentBody'><div class='typing' id='typing'><i></i><i></i><i></i></div></div><div class='quick' id='quick'></div></div><div class='card plot'>{figs['price']}</div></div></section>
<footer class='footer'>Dashboard generado automáticamente con Python, yfinance/scikit-learn y Plotly. Diseño magIA para comunicación ejecutiva.</footer>
<script>
const tabs=document.querySelectorAll('.tab'); const sections=document.querySelectorAll('.section'); tabs.forEach(t=>t.onclick=()=>{{tabs.forEach(x=>x.classList.remove('active'));sections.forEach(s=>s.classList.remove('active'));t.classList.add('active');document.getElementById(t.dataset.tab).classList.add('active');}});
const messages={agent_json}; const quick={quick_json}; const body=document.getElementById('agentBody'); const typing=document.getElementById('typing');
function addMsg(title,text,delay=0){{setTimeout(()=>{{typing.style.display='none'; const div=document.createElement('div'); div.className='msg'; div.innerHTML=`<strong>${{title}}</strong><p>${{text}}</p>`; body.appendChild(div); body.scrollTop=body.scrollHeight;}},delay);}}
messages.forEach((m,i)=>{{setTimeout(()=>{{typing.style.display='flex';}}, i*1400); addMsg(m.title,m.text,i*1400+650);}});
const q=document.getElementById('quick'); Object.keys(quick).forEach(k=>{{const b=document.createElement('button'); b.textContent=k; b.onclick=()=>{{const user=document.createElement('div'); user.className='msg'; user.style.marginLeft='auto'; user.style.background='rgba(155,92,255,.16)'; user.innerHTML=`<strong>Pregunta ejecutiva</strong><p>${{k}}</p>`; body.appendChild(user); typing.style.display='flex'; body.appendChild(typing); addMsg('Respuesta del agente',quick[k],650);}}; q.appendChild(b);}});
</script>
</body>
</html>
"""

out_path = os.path.join(BASE_DIR, 'dashboard_robot_fx_AVANZADO.html')
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html)
print('Dashboard guardado en:', out_path)
display(HTML(f"<a href='{out_path}' target='_blank'>Abrir dashboard avanzado</a>"))
try:
    display(IFrame(out_path, width='100%', height=780))
except Exception:
    pass
